# Question 2: Visa Category and Chargeability Area Comparison

**How do movement patterns differ across family-sponsored visa categories and chargeability areas?**

The analysis uses the full transformed Visa Bulletin dataset. Movement patterns are compared across visa categories and chargeability areas, with Final Action Dates and Dates for Filing evaluated separately where appropriate.

The main statistical approach for this question is the Chi-square test of independence to examine whether movement status is associated with visa category or chargeability area.


## Hypotheses for Research Question 2

### Visa Category and Movement Status
- **H0:** Movement status is independent of visa category.
- **H1:** Movement status is associated with visa category.

### Chargeability Area and Movement Status
- **H0:** Movement status is independent of chargeability area.
- **H1:** Movement status is associated with chargeability area.

These hypotheses are tested separately for **Final Action Dates** and **Dates for Filing**.

In [6]:
import pandas as pd

q2_df = pd.read_csv(
    "../data/processed/visa_bulletins_transformed_full.csv",
    parse_dates=["bulletin_date", "cutoff_date"]
)

q2_df.head()

,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value,cutoff_status,cutoff_date,days_behind_bulletin,movement_days,movement_status,months_stagnant,current_streak
0,2016-10-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2130,NaN,NaN,0,0
1,2016-11-30,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2160,0.0,No Movement,1,0
2,2016-12-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2191,0.0,No Movement,2,0
3,2017-01-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2222,0.0,No Movement,3,0
4,2017-02-28,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2250,0.0,No Movement,4,0


In [7]:
print("Shape:", q2_df.shape)
print("Visa categories:", q2_df["visa_category"].unique())
print("Chargeability areas:", q2_df["chargeability_area"].unique())
print("Table types:", q2_df["table_type"].unique())
print("Movement status counts:")
print(q2_df["movement_status"].value_counts(dropna=False))

Shape: (6000, 12)
Visa categories: <StringArray>
['F1', 'F2A', 'F2B', 'F3', 'F4']
Length: 5, dtype: str
Chargeability areas: <StringArray>
['All Chargeability Areas Except Those Listed',
                                       'China',
                                       'India',
                                      'Mexico',
                                 'Philippines']
Length: 5, dtype: str
Table types: <StringArray>
['Dates for Filing', 'Final Action']
Length: 2, dtype: str
Movement status counts:
movement_status
No Movement    3068
Advance        2852
NaN              50
Retrogress       30
Name: count, dtype: int64


In [8]:
#separate the data into two dataframes based on the table type
final_action_q2 = q2_df[
    q2_df["table_type"] == "Final Action"].copy()

dates_for_filing_q2 = q2_df[
    q2_df["table_type"] == "Dates for Filing"].copy()

print("Final Action Dates shape:", final_action_q2.shape)
print("Dates for Filing shape:", dates_for_filing_q2.shape)


Final Action Dates shape: (3000, 12)
Dates for Filing shape: (3000, 12)


In [9]:
# Remove rows where movement status cannot be calculated
# because there is no previous month for comparison
#drona does not effect the next month since values already exist for the next month, so we can drop these rows

final_action_q2 = final_action_q2.dropna(
    subset=["movement_status"]
)

dates_for_filing_q2 = dates_for_filing_q2.dropna(
    subset=["movement_status"]
)

In [10]:
print("Final Action rows:", len(final_action_q2))
print("Dates for Filing rows:", len(dates_for_filing_q2))

print("\nFinal Action movement counts:")
print(final_action_q2["movement_status"].value_counts())

print("\nDates for Filing movement counts:")
print(dates_for_filing_q2["movement_status"].value_counts())

Final Action rows: 2975
Dates for Filing rows: 2975

Final Action movement counts:
movement_status
Advance        1737
No Movement    1208
Retrogress       30
Name: count, dtype: int64

Dates for Filing movement counts:
movement_status
No Movement    1860
Advance        1115
Name: count, dtype: int64


In [11]:
# Compare movement status across visa categories for Final Action

final_action_category = pd.crosstab(
    final_action_q2["visa_category"],
    final_action_q2["movement_status"]
)

final_action_category

movement_status,Advance,No Movement,Retrogress
visa_category,,,
F1,303,284,8
F2A,458,126,11
F2B,323,271,1
F3,343,248,4
F4,310,279,6


In [12]:
filing_category = pd.crosstab(
    dates_for_filing_q2["visa_category"],
    dates_for_filing_q2["movement_status"]
)

filing_category

movement_status,Advance,No Movement
visa_category,,
F1,188,407
F2A,350,245
F2B,194,401
F3,193,402
F4,190,405


## Statistical Analysis

The previous steps prepared and validated the Q2 dataset by separating Final Action Dates and Dates for Filing and confirming the movement-status distributions.

The next step uses Chi-square tests of independence to examine whether movement status is associated with visa category or chargeability area. Final Action Dates and Dates for Filing are tested separately because their movement patterns differ.

In [13]:
%pip install scipy 



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from scipy.stats import chi2_contingency

### Final Action: Visa Category and Movement Status

In [ ]:
# Create a contingency table for visa category and movement status
# using only Final Action data.
# reference point 

final_action_category_table = pd.crosstab(
    final_action_q2["visa_category"],
    final_action_q2["movement_status"]
)

final_action_category_table

movement_status,Advance,No Movement,Retrogress
visa_category,,,
F1,303,284,8
F2A,458,126,11
F2B,323,271,1
F3,343,248,4
F4,310,279,6


In [ ]:
# Chi-square test:
# Checks whether Final Action movement status is associated with visa category
#movement status x visa category

chi2, p_value, dof, expected = chi2_contingency(
    final_action_category_table
)

print("Chi-square statistic:", round(chi2, 2))
print("p-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square statistic: 128.64
p-value: 5.418527845483217e-24
Degrees of freedom: 8


### Interpretation 1.0

The Chi-square test showed a statistically significant association between visa category and movement status for Final Action Dates
(X^2 = 128.64, df = 8, p < 0.001 )

Because the p-value is less than 0.05, we reject the null hypothesis that movement status is independent of visa category.

This indicates that the distribution of Advance, No Movement, and Retrogress outcomes differs across the family-sponsored visa categories.

F2A appears especially different, with a larger proportion of Advance observations compared with the other categories..

### Final Action: Chargeability Area and Movement Status

In [ ]:
# Create a contingency table for chargeability area and movement status
# using only Final Action data.
# reference point 

final_action_area_table = pd.crosstab(
    final_action_q2["chargeability_area"],
    final_action_q2["movement_status"]
)

final_action_area_table

movement_status,Advance,No Movement,Retrogress
chargeability_area,,,
All Chargeability Areas Except Those Listed,352,236,7
China,352,236,7
India,354,235,6
Mexico,340,249,6
Philippines,339,252,4


In [18]:
# Chi-square test:
# Checks whether Final Action movement status is associated with chargeability areas
# movement status x chargeability areas

chi2, p_value, dof, expected = chi2_contingency(
    final_action_area_table
)

print("Chi-square statistic:", round(chi2, 2))
print("p-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square statistic: 2.72
p-value: 0.9505649348707803
Degrees of freedom: 8


### Interpretation 1.1

The Chi-square test did not show a statistically significant association between chargeability area and movement status for Final Action Dates 
(X^2 = 2.72, df = 8, p = 0.951).

Because the p-value is greater than 0.05, we fail to reject the null hypothesis that movement status is independent of chargeability area.

This suggests that the distribution of Advance, No Movement, and Retrogress outcomes does not differ significantly across chargeability areas in the Final Action table.


### Dates for Filing: Visa Category and Movement Status

In [19]:
# Create a contingency table for visa category and movement status
# using only Dates for Filing data.
#reference point

filing_category_table = pd.crosstab(
    dates_for_filing_q2["visa_category"],
    dates_for_filing_q2["movement_status"]
)

filing_category_table

movement_status,Advance,No Movement
visa_category,,
F1,188,407
F2A,350,245
F2B,194,401
F3,193,402
F4,190,405


In [20]:
# Chi-square test:
# Checks whether Dates for Filing movement status is associated with visa category
# movement status x visa category

chi2, p_value, dof, expected = chi2_contingency(
    filing_category_table
)

print("Chi-square statistic:", round(chi2, 2))
print("p-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square statistic: 144.77
p-value: 2.6875493816101774e-30
Degrees of freedom: 4


### Interpretation 2.0

The Chi-square test showed a statistically significant association between visa category and movement status for Dates for Filing 
(X^2 = 144.77, df = 4, p < 0.001).

Because the p-value is less than 0.05, we reject the null hypothesis that movement status is independent of visa category.

This indicates that the distribution of Advance and No Movement outcomes differs across the family-sponsored visa categories in the Dates for Filing table.

### Dates for Filing: Chargeability Area and Movement Status

In [21]:
# Create a contingency table for chargeability area and movement status
# using only Dates for Filing data.
#reference point

filing_area_table = pd.crosstab(
    dates_for_filing_q2["chargeability_area"],
    dates_for_filing_q2["movement_status"]
)

filing_area_table

movement_status,Advance,No Movement
chargeability_area,,
All Chargeability Areas Except Those Listed,229,366
China,229,366
India,222,373
Mexico,226,369
Philippines,209,386


In [22]:
# Chi-square test:
# Checks whether Dates for Filing movement status is associated with chargeability area.
# movement status x chargeability areas

chi2, p_value, dof, expected = chi2_contingency(
    filing_area_table
)

print("Chi-square statistic:", round(chi2, 2))
print("p-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square statistic: 1.99
p-value: 0.7368719736036833
Degrees of freedom: 4


### Interpretation 2.1
The Chi-square test did not show a statistically significant association between chargeability area and movement status for Dates for Filing 
(X^2 = 1.99, df = 4, p = 0.737).

Because the p-value is greater than 0.05, we fail to reject the null hypothesis that movement status is independent of chargeability area.

This suggests that the distribution of Advance and No Movement outcomes does not differ significantly across chargeability areas in the Dates for Filing table.

### Overall Interpretation

Final Action |	Visa Category × Movement	| Significant **[p-value < alpha ]**

Final Action | Chargeability Area × Movement | Not significant **[p-value >= alpha ]**

Dates for Filing | Visa Category × Movement | Significant **[p-value < alpha ]**

Dates for Filing | Chargeability Area × Movement | Not significant **[p-value >= alpha ]**


**Overall, the Chi-square results provide evidence that movement status is associated with visa category in both Final Action Dates and Dates for Filing, but there is no statistically significant evidence of an association between movement status and chargeability area.**

## Preliminary Findings for Question 2

Movement patterns differed across visa categories in both Final Action Dates and Dates for Filing. The Chi-square tests showed a statistically significant association between visa category and movement status for both table types.

In the descriptive results, F2A showed a stronger Advance pattern than the other family-sponsored categories. This supports the statistical finding that movement outcomes are not distributed equally across visa categories.

Chargeability areas showed some visible differences in the descriptive analysis, particularly for Mexico and the Philippines. However, the Chi-square tests did not find a statistically significant association between chargeability area and movement status for either Final Action Dates or Dates for Filing. **Further research might be needed to examine differences within individual visa categories.**

Overall, the results suggest that visa category is associated with movement status, while no statistically significant association was found between chargeability area and movement status during the study period.